In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
import rioxarray


# ============================================================
# PATHS
# ============================================================

DATA_0 = Path("data_0.nc")
DATA_1 = Path("data_1.nc")

OUTPUT_DIR = Path("derived")
OUTPUT_DIR.mkdir(exist_ok=True)

RASTER_DIR = OUTPUT_DIR / "rasters"
RASTER_DIR.mkdir(exist_ok=True)


# ============================================================
# 1. LOAD RAW ERA5-LAND FILES
# ============================================================

ds = xr.open_dataset(DATA_0)
ds2 = xr.open_dataset(DATA_1)


# ============================================================
# 2. NORMALIZE LONGITUDES IN data_1.nc
#
# data_0 uses approximately:
#     -118 ... -111
#
# data_1 uses:
#      242 ... 249
#
# These describe the same locations.
# ============================================================

ds2 = ds2.assign_coords(
    longitude=((ds2.longitude + 180) % 360) - 180
).sortby("longitude")


# ============================================================
# 3. VERIFY THAT BOTH FILES REALLY DESCRIBE THE SAME GRID
# ============================================================

assert np.allclose(ds.latitude.values, ds2.latitude.values)
assert np.allclose(ds.longitude.values, ds2.longitude.values)
assert np.array_equal(ds.valid_time.values, ds2.valid_time.values)

print("Grid checks passed.")


# ============================================================
# 4. REMOVE TINY FLOATING-POINT COORDINATE DIFFERENCES
#
# We use data_0's coordinates as the canonical coordinates.
# ============================================================

ds2 = ds2.assign_coords(
    latitude=ds.latitude,
    longitude=ds.longitude,
    valid_time=ds.valid_time,
)


# ============================================================
# 5. MERGE THE TWO DATASETS
#
# join='exact' makes the operation fail rather than silently
# generating duplicate longitude cells.
# ============================================================

weather = xr.merge(
    [ds, ds2],
    join="exact",
    compat="no_conflicts",
)

assert weather.sizes["valid_time"] == 744
assert weather.sizes["latitude"] == 51
assert weather.sizes["longitude"] == 71

print("Merged dimensions:")
print(weather.sizes)


# ============================================================
# 6. DERIVED METEOROLOGICAL VARIABLES
# ============================================================

# ---- Air temperature: Kelvin -> Celsius
weather["t2m_c"] = weather["t2m"] - 273.15
weather["t2m_c"].attrs = {
    "long_name": "2 metre air temperature",
    "units": "degC",
}

# ---- Temperature of snow layer: Kelvin -> Celsius
weather["tsn_c"] = weather["tsn"] - 273.15
weather["tsn_c"].attrs = {
    "long_name": "Temperature of snow layer",
    "units": "degC",
}

# ---- Wind speed from U and V components
weather["wind10"] = np.hypot(
    weather["u10"],
    weather["v10"],
)
weather["wind10"].attrs = {
    "long_name": "10 metre wind speed",
    "units": "m s-1",
}

# ---- Remove tiny negative floating-point noise
weather["sde"] = weather["sde"].clip(min=0)

# ---- Constrain snow-cover percentage to physical range
weather["snowc"] = weather["snowc"].clip(min=0, max=100)


# ============================================================
# 7. ADD SPATIAL METADATA
#
# ERA5-Land is on a longitude/latitude WGS84-style grid.
# This makes GIS software much less likely to misinterpret it.
# ============================================================

weather = weather.rio.set_spatial_dims(
    x_dim="longitude",
    y_dim="latitude",
)

weather = weather.rio.write_crs("EPSG:4326")


# ============================================================
# 8. EXPORT THE CLEANED HOURLY DATASET
# ============================================================

encoding = {}

for var in weather.data_vars:
    if np.issubdtype(weather[var].dtype, np.number):
        encoding[var] = {
            "zlib": True,
            "complevel": 4,
        }

weather_nc_path = OUTPUT_DIR / "weather_montana_2020_01.nc"

weather.to_netcdf(
    weather_nc_path,
    encoding=encoding,
)

print(f"\nSaved hourly NetCDF:")
print(weather_nc_path)


# ============================================================
# 9. DERIVE JANUARY SUMMARY MAPS
# ============================================================

# Mean air temperature during January
jan_mean_temp = weather["t2m_c"].mean(dim="valid_time")
jan_mean_temp.name = "jan_mean_temperature"
jan_mean_temp.attrs["units"] = "degC"

# Coldest hourly temperature reached during January
jan_min_temp = weather["t2m_c"].min(dim="valid_time")
jan_min_temp.name = "jan_min_temperature"
jan_min_temp.attrs["units"] = "degC"

# Number of hourly observations below freezing
freezing_hours = (weather["t2m_c"] < 0).sum(dim="valid_time")
freezing_hours.name = "freezing_hours"
freezing_hours.attrs = {
    "long_name": "Hours with 2 metre air temperature below 0 degC",
    "units": "hours",
}

# Mean wind speed
jan_mean_wind = weather["wind10"].mean(dim="valid_time")
jan_mean_wind.name = "jan_mean_wind10"
jan_mean_wind.attrs["units"] = "m s-1"

# 95th percentile wind speed
jan_p95_wind = weather["wind10"].quantile(
    0.95,
    dim="valid_time"
)

# quantile() may leave a scalar 'quantile' coordinate;
# remove it for simpler GIS output.
if "quantile" in jan_p95_wind.coords:
    jan_p95_wind = jan_p95_wind.drop_vars("quantile")

jan_p95_wind.name = "jan_p95_wind10"
jan_p95_wind.attrs = {
    "long_name": "95th percentile 10 metre wind speed",
    "units": "m s-1",
}

# Maximum snow depth
jan_max_snow_depth = weather["sde"].max(dim="valid_time")
jan_max_snow_depth.name = "jan_max_snow_depth"
jan_max_snow_depth.attrs["units"] = "m"

# Mean snow cover percentage
jan_mean_snow_cover = weather["snowc"].mean(dim="valid_time")
jan_mean_snow_cover.name = "jan_mean_snow_cover"
jan_mean_snow_cover.attrs["units"] = "%"

# Hours during which >= 50% of the ERA5 grid cell was snow-covered
snow_cover_hours_50 = (
    weather["snowc"] >= 50
).sum(dim="valid_time")

snow_cover_hours_50.name = "snow_cover_hours_ge50"
snow_cover_hours_50.attrs = {
    "long_name": "Hours with at least 50 percent snow cover",
    "units": "hours",
}


# ============================================================
# 10. HELPER FUNCTION FOR QGIS-READY GEOTIFF EXPORT
# ============================================================

def export_geotiff(data_array, filename):
    """
    Export a 2-D ERA5-derived DataArray as an EPSG:4326 GeoTIFF.
    QGIS can reproject this on the fly into the EPSG:5070 project.
    """

    da = data_array.copy()

    da = da.rio.set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
    )

    da = da.rio.write_crs("EPSG:4326")

    output_path = RASTER_DIR / filename

    da.rio.to_raster(
        output_path,
        compress="DEFLATE",
    )

    print(f"Saved: {output_path}")


# ============================================================
# 11. EXPORT SUMMARY RASTERS
# ============================================================

export_geotiff(
    jan_mean_temp,
    "jan_2020_mean_temperature_c.tif",
)

export_geotiff(
    jan_min_temp,
    "jan_2020_min_temperature_c.tif",
)

export_geotiff(
    freezing_hours,
    "jan_2020_freezing_hours.tif",
)

export_geotiff(
    jan_mean_wind,
    "jan_2020_mean_wind10_ms.tif",
)

export_geotiff(
    jan_p95_wind,
    "jan_2020_p95_wind10_ms.tif",
)

export_geotiff(
    jan_max_snow_depth,
    "jan_2020_max_snow_depth_m.tif",
)

export_geotiff(
    jan_mean_snow_cover,
    "jan_2020_mean_snow_cover_pct.tif",
)

export_geotiff(
    snow_cover_hours_50,
    "jan_2020_snow_cover_hours_ge50pct.tif",
)


# ============================================================
# 12. SIMPLE DOMAIN-WIDE DIAGNOSTICS
# ============================================================

print("\n--- DOMAIN-WIDE JANUARY 2020 DIAGNOSTICS ---")

print(
    f"Temperature min:  "
    f"{weather['t2m_c'].min().item():.2f} °C"
)

print(
    f"Temperature mean: "
    f"{weather['t2m_c'].mean().item():.2f} °C"
)

print(
    f"Temperature max:  "
    f"{weather['t2m_c'].max().item():.2f} °C"
)

print(
    f"Maximum wind:     "
    f"{weather['wind10'].max().item():.2f} m/s"
)

print(
    f"Maximum snow depth: "
    f"{weather['sde'].max().item():.2f} m"
)

print(
    f"Snow cover range: "
    f"{weather['snowc'].min().item():.1f}"
    f"–{weather['snowc'].max().item():.1f} %"
)

print("\nProcessing complete.")

Grid checks passed.
Merged dimensions:
Frozen({'valid_time': 744, 'latitude': 51, 'longitude': 71})

Saved hourly NetCDF:
derived/weather_montana_2020_01.nc
Saved: derived/rasters/jan_2020_mean_temperature_c.tif
Saved: derived/rasters/jan_2020_min_temperature_c.tif
Saved: derived/rasters/jan_2020_freezing_hours.tif
Saved: derived/rasters/jan_2020_mean_wind10_ms.tif
Saved: derived/rasters/jan_2020_p95_wind10_ms.tif
Saved: derived/rasters/jan_2020_max_snow_depth_m.tif
Saved: derived/rasters/jan_2020_mean_snow_cover_pct.tif
Saved: derived/rasters/jan_2020_snow_cover_hours_ge50pct.tif

--- DOMAIN-WIDE JANUARY 2020 DIAGNOSTICS ---
Temperature min:  -33.19 °C
Temperature mean: -4.87 °C
Temperature max:  16.25 °C
Maximum wind:     14.76 m/s
Maximum snow depth: 1.88 m
Snow cover range: 0.0–100.0 %

Processing complete.
